# SMS Spam Classifier - Full Pipeline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('spam.csv', encoding='latin-1')[['v1','v2']]
df.columns = ['label','message']
df['length'] = df['message'].apply(len)
print(df['label'].value_counts())
sns.countplot(x='label', data=df); plt.show()

In [ ]:
import re, string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lem = WordNetLemmatizer(); sw = set(stopwords.words('english'))
def clean(t):
    t = t.lower(); t = re.sub(r'http\S+|www\S+','',t); t = re.sub(r'\d+','',t)
    t = t.translate(str.maketrans('','',string.punctuation))
    return ' '.join(lem.lemmatize(w) for w in t.split() if w not in sw)
df['clean'] = df['message'].apply(clean)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

Xtr, Xte, ytr, yte = train_test_split(df['clean'], (df['label']=='spam').astype(int), test_size=0.2, random_state=42, stratify=(df['label']=='spam').astype(int))

models = {
 'NB': MultinomialNB(),
 'LR': LogisticRegression(max_iter=1000),
 'SVM': LinearSVC(),
}
for name, m in models.items():
    pipe = Pipeline([('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2))), ('clf', m)])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    print('==', name); print(classification_report(yte, pred, target_names=['ham','spam']))

In [ ]:
import joblib
best = Pipeline([('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2))), ('clf', LogisticRegression(max_iter=1000))])
best.fit(Xtr, ytr)
joblib.dump(best, 'spam_pipeline.joblib')
print(best.predict(['Free prize win now','See you at 5']))